# Real-Time Web Search with Llama Using free-web-search-ultimate

This notebook demonstrates how to give Llama models real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no additional API keys required**.

[![free-web-search-ultimate](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a tool for Llama models
- How to build a research assistant with real-time web access
- How to handle tool use with Llama's function calling API
- How to combine web search with Llama's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool (`free-web-search`) |

In [ ]:
# Install required packages
%pip install free-web-search-ultimate groq --quiet
# Note: This example uses Groq for fast Llama inference, but you can use any Llama API

In [ ]:
import json
import os
from groq import Groq
from free_web_search import search, search_news

# Initialize the Groq client (for fast Llama inference)
# Get your free API key from https://console.groq.com/
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Use this when you need up-to-date information that may not be in your training data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up on the web"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "news_search",
            "description": "Search for recent news articles. Use this when you need the latest news on a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The news topic to search for"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of news articles to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool and return the result as a string."""
    try:
        if tool_name == "web_search":
            results = search(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        elif tool_name == "news_search":
            results = search_news(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        else:
            return f"Unknown tool: {tool_name}"
        
        if not results:
            return "No results found."
        
        formatted = []
        for i, result in enumerate(results, 1):
            title = result.get('title', 'No title')
            url = result.get('url', result.get('href', 'No URL'))
            body = result.get('body', result.get('snippet', 'No description'))
            formatted.append(f"{i}. {title}\n   URL: {url}\n   {body}")
        
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error: {str(e)}"

print("Setup complete!")

## Build the Research Assistant

In [ ]:
def research_with_llama(question: str, model: str = "llama-3.3-70b-versatile") -> str:
    """
    Use Llama with web search tools to answer a question.
    
    Args:
        question: The question to research
        model: The Llama model to use
    
    Returns:
        The final answer from Llama
    """
    print(f"Question: {question}\n")
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful research assistant with access to real-time web search. "
                "Use the web_search and news_search tools to find current information when needed. "
                "Always cite your sources by including the URLs from search results."
            )
        },
        {"role": "user", "content": question}
    ]
    
    # Agentic loop
    max_iterations = 5
    for i in range(max_iterations):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        # If no tool calls, we have the final answer
        if not message.tool_calls:
            return message.content
        
        # Execute tool calls
        messages.append(message)
        
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            
            print(f"Calling tool: {tool_name}({tool_input})")
            tool_result = execute_tool(tool_name, tool_input)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })
    
    return "Max iterations reached."

print("Research assistant ready!")

## Example: Research with Llama

In [ ]:
# Example: Ask about current AI developments
answer = research_with_llama(
    "What are the latest developments in open-source LLMs this week?"
)
print("\n--- Final Answer ---")
print(answer)

## Direct Usage

You can also use the package directly without any LLM.

In [ ]:
from free_web_search import search, search_news

# Direct web search
results = search("Meta Llama latest updates", max_results=3)
for r in results:
    print(f"Title: {r.get('title', 'N/A')}")
    print(f"URL: {r.get('url', r.get('href', 'N/A'))}")
    print()

## MCP Server Usage

The `free-web-search-ultimate` package also works as an MCP server:

```bash
# Install and run as MCP server
pip install free-web-search-ultimate
free-web-search-mcp

# Or use as CLI
free-web-search "latest Llama news"
```

## Summary

In this notebook, we demonstrated how to:

1. **Install** `free-web-search-ultimate` — a zero-cost, privacy-first web search package
2. **Define function tools** for Llama models using the OpenAI-compatible API
3. **Build a research assistant** with an agentic tool-use loop
4. **Research complex topics** by combining Llama's reasoning with real-time web search

### Resources

- [free-web-search-ultimate on GitHub](https://github.com/wd041216-bit/free-web-search-ultimate)
- [free-web-search-ultimate on PyPI](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [Llama Function Calling Documentation](https://llama.meta.com/docs/model-cards-and-prompt-formats/llama3_1/)